# Day 27: Building a Simple Chatbot (LLM-Powered)

On Day 26 our Chai Wala bot answered **one** question. Today it becomes a real **chat**: it remembers what the customer said, and it gets a **web page**.

## 🎯 The example we will build

| Turn | Customer | Bot |
|---|---|---|
| 1 | *How much is a samosa?* | A samosa is Rs 60. |
| 2 | *I want 3. What is the total?* | 3 samosas are Rs 180. |

In turn 2 the customer does not say "samosa". The bot must **remember** turn 1.

## 📋 Agenda

```
TOP     Step 5   Streamlit web page       a chat app you can show
          ▲
        Step 4   A chat loop              type, answer, repeat
          ▲
        Step 3   Keep it short            stay under the context limit
          ▲
        Step 2   Remember the chat        the history list
          ▲
BOTTOM  Step 1   Start from Day 26        the same good prompt
```

### New words today

| Word | Meaning |
|---|---|
| **Turn** | One message in the chat (Day 24). |
| **Multi-turn** | A chat with many turns. The bot must remember the old ones. |
| **History** | The list of all messages so far. |
| **Loop** | Code that repeats: read a message, answer, read the next one. |
| **Streamlit** | A Python library that makes a web page from a small Python file. |
| **Session state** | Streamlit's memory. It keeps our history between messages. |

---
# Class Problem: a Chai Wala chat

## Step 1: Start from Day 26

The same key, the same `ask` function and the same 5-part prompt.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv          # reads secrets from the .env file
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free)

load_dotenv()                                       # finds .env and loads OPEN_ROUTER_KEY
key = os.getenv("OPEN_ROUTER_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

def ask(messages, temperature=0):
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:                    # no key? rate limit (429)? wrong model name?
        return f"ERROR: {e}"

In [ ]:
FACTS = """
Menu: doodh patti chai Rs 120, Kashmiri chai Rs 200, green tea Rs 100,
plain paratha Rs 80, aloo paratha Rs 150, samosa Rs 60, cake rusk pack Rs 250.
Delivery: Gulberg, DHA, Model Town, Johar Town. Charge Rs 100, free for orders of Rs 1500 or more.
Delivery takes 30 to 45 minutes. Last order at 10:30 pm.
Timings: every day 9 am to 11 pm. Friday closed 1 pm to 2 pm for Jumma.
Payment: cash, card, JazzCash, Easypaisa. Cash on delivery is available.
Phone: 042-1234567. WhatsApp: 0300-1234567.
"""

SYSTEM_PROMPT = f"""
You are the support bot of Chai Wala, a tea shop in Gulberg, Lahore.

Facts:
{FACTS}

Rules:
- Use ONLY the facts above. Never make up a price, place or time.
- Write prices like "Rs 120".
- Be friendly. Reply in at most 3 short sentences.
- If the answer is not in the facts, say: "Sorry, I don't know. Please call 042-1234567."
"""

## Step 2: Remember the chat

**Remember Day 24:** the API has **no memory**. Each request is new. So **we** keep the chat in a list called `history`, and we send the **whole list** every time.

The `chat` function does 4 things:

1. Add the customer message to `history`.
2. Send `history` to the model.
3. Add the bot reply to `history`.
4. Return the reply.

In [ ]:
history = [{"role": "system", "content": SYSTEM_PROMPT}]      # the chat starts with the rules

def chat(question):
    history.append({"role": "user", "content": question})        # 1
    reply = ask(history)                                          # 2
    history.append({"role": "assistant", "content": reply})      # 3
    return reply                                                  # 4

print(chat("How much is a samosa?"))
print(chat("I want 3. What is the total?"))     # it remembers "samosa" from turn 1

In [ ]:
for m in history:                           # look inside the history
    print(m["role"], ":", m["content"][:60])

**✏️ Quick Check:** Tell the bot *"My name is Ali."*, then ask *"What is my name?"*. Does it know? Why?

In [ ]:
# your code here

**Answer:** Yes, it says Ali. *"My name is Ali"* is in `history`, and we send the whole `history` every time.

In [ ]:
print(chat("My name is Ali."))
print(chat("What is my name?"))

## Step 3: Keep it short

- Every turn makes `history` **longer**. More tokens = more cost and slower answers (Day 24).
- Every model has a **context limit**. A very long chat gives an error.
- **Fix:** send the **rules** (`history[0]`) + only the **last 6 messages**. The full history stays in the list. We only send a short copy.

In [ ]:
def chat(question):
    history.append({"role": "user", "content": question})
    short = [history[0]] + history[1:][-6:]                       # rules + last 6 messages
    reply = ask(short)
    history.append({"role": "assistant", "content": reply})
    return reply

print(len(history), "messages in history")
print(len([history[0]] + history[1:][-6:]), "messages sent")      # never more than 7

**✏️ Quick Check:** The chat has 50 messages. The customer said their name in message 2. Can the bot still remember the name? Why?

In [ ]:
# your code here

**Answer:** No. We only send the rules + the last 6 messages, so message 2 is not sent. This is the price of a short history. (A fix: save important facts, like the name, in the system prompt.)

## Step 4: A chat loop

A real chat goes on and on. A `while` loop reads a message, answers, and repeats. Type **bye** to stop.

In [ ]:
history = [{"role": "system", "content": SYSTEM_PROMPT}]      # start a new chat

while True:
    question = input("You: ")
    if question.lower() == "bye":
        print("Bot: Allah Hafiz! Come again for chai.")
        break
    print("Bot:", chat(question))

## Step 5: Streamlit web page

A chat in a notebook is only for us. A **web page** is something you can **show** to people.

1. Install once: `pip install streamlit`
2. Run the cell below. It **writes** a file named `chai_wala_app.py`.
3. Open a **terminal** in this folder and run: `streamlit run chai_wala_app.py`
4. The browser opens. Chat with your bot!

**Important idea:** Streamlit runs the **whole file again** every time you send a message. Normal variables are lost. So we keep the history in **`st.session_state`**, Streamlit's memory.

| Streamlit line | What it does |
|---|---|
| `st.title(...)` | The big heading on the page. |
| `st.session_state.messages` | Our `history`. It stays between messages. |
| `st.chat_message(role).write(text)` | Shows one chat bubble (user or assistant). |
| `st.chat_input(...)` | The text box at the bottom. It gives the text, or `None` if nothing was sent. |

In [ ]:
%%writefile chai_wala_app.py
import os
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPEN_ROUTER_KEY"))
MODEL = "nvidia/nemotron-3.5-lightning:free"

FACTS = """
Menu: doodh patti chai Rs 120, Kashmiri chai Rs 200, green tea Rs 100,
plain paratha Rs 80, aloo paratha Rs 150, samosa Rs 60, cake rusk pack Rs 250.
Delivery: Gulberg, DHA, Model Town, Johar Town. Charge Rs 100, free for orders of Rs 1500 or more.
Delivery takes 30 to 45 minutes. Last order at 10:30 pm.
Timings: every day 9 am to 11 pm. Friday closed 1 pm to 2 pm for Jumma.
Payment: cash, card, JazzCash, Easypaisa. Cash on delivery is available.
Phone: 042-1234567. WhatsApp: 0300-1234567.
"""

SYSTEM_PROMPT = f"""
You are the support bot of Chai Wala, a tea shop in Gulberg, Lahore.

Facts:
{FACTS}

Rules:
- Use ONLY the facts above. Never make up a price, place or time.
- Write prices like "Rs 120".
- Be friendly. Reply in at most 3 short sentences.
- If the answer is not in the facts, say: "Sorry, I don't know. Please call 042-1234567."
"""

st.title("☕ Chai Wala Support Bot")

if "messages" not in st.session_state:                       # 1. first run only: start the history
    st.session_state.messages = [{"role": "system", "content": SYSTEM_PROMPT}]

for m in st.session_state.messages[1:]:                       # 2. show the old chat (not the rules)
    st.chat_message(m["role"]).write(m["content"])

question = st.chat_input("Ask Chai Wala...")                  # 3. the text box at the bottom
if question:
    st.chat_message("user").write(question)
    st.session_state.messages.append({"role": "user", "content": question})

    history = st.session_state.messages
    short = [history[0]] + history[1:][-6:]                   # 4. rules + last 6 messages
    r = client.chat.completions.create(model=MODEL, messages=short, temperature=0)
    reply = r.choices[0].message.content

    st.chat_message("assistant").write(reply)
    st.session_state.messages.append({"role": "assistant", "content": reply})

**✏️ Quick Check:** Add a small grey line under the title: *"Ask about our menu, delivery and timings."* Use `st.caption(...)`.

In [ ]:
# your code here

**Answer:** Add this line after `st.title(...)` in `chai_wala_app.py`, save, and click **Rerun** in the browser:

```python
st.caption("Ask about our menu, delivery and timings.")
```

## ✅ Complete code (Class Problem)

The notebook chat in one cell. (The web page is the file `chai_wala_app.py` from Step 5.)

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv          # reads secrets from the .env file
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free)

load_dotenv()                                       # finds .env and loads OPEN_ROUTER_KEY
key = os.getenv("OPEN_ROUTER_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

def ask(messages, temperature=0):
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:                    # no key? rate limit (429)? wrong model name?
        return f"ERROR: {e}"

FACTS = """
Menu: doodh patti chai Rs 120, Kashmiri chai Rs 200, green tea Rs 100,
plain paratha Rs 80, aloo paratha Rs 150, samosa Rs 60, cake rusk pack Rs 250.
Delivery: Gulberg, DHA, Model Town, Johar Town. Charge Rs 100, free for orders of Rs 1500 or more.
Delivery takes 30 to 45 minutes. Last order at 10:30 pm.
Timings: every day 9 am to 11 pm. Friday closed 1 pm to 2 pm for Jumma.
Payment: cash, card, JazzCash, Easypaisa. Cash on delivery is available.
Phone: 042-1234567. WhatsApp: 0300-1234567.
"""

SYSTEM_PROMPT = f"""
You are the support bot of Chai Wala, a tea shop in Gulberg, Lahore.

Facts:
{FACTS}

Rules:
- Use ONLY the facts above. Never make up a price, place or time.
- Write prices like "Rs 120".
- Be friendly. Reply in at most 3 short sentences.
- If the answer is not in the facts, say: "Sorry, I don't know. Please call 042-1234567."
"""

history = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(question):
    history.append({"role": "user", "content": question})
    short = [history[0]] + history[1:][-6:]          # rules + last 6 messages
    reply = ask(short)
    history.append({"role": "assistant", "content": reply})
    return reply

print(chat("How much is a samosa?"))
print(chat("I want 3. What is the total?"))

### Key takeaways

- The API has no memory. **We** keep the chat in a `history` list and send it every time.
- To stay under the context limit, send the **rules + the last few messages**.
- A `while` loop with `input()` makes a chat in the terminal.
- **Streamlit** turns about 20 lines of Python into a chat web page.
- Streamlit runs the whole file on every message, so the history lives in **`st.session_state`**.

---
# 🏠 Home Problem

Make the Chai Wala app better. Start from a copy: `chai_wala_app.py` → `chai_wala_app_home.py`.

1. **Welcome message:** when the chat starts, the bot says first: *"Assalam o Alaikum! Welcome to Chai Wala. Ask me about our menu, delivery or timings."* (Hint: add it to `st.session_state.messages` as an `assistant` message, in the same place where you add the system message.)
2. **New chat button:** in the sidebar, add a button `st.sidebar.button("New chat")`. When it is clicked, start the history again.
3. **Test memory** in the browser: ask *"How much is Kashmiri chai?"*, then *"And for 2 cups?"*.

**Project piece:** make a second app for **your own** project. Use your Day 26 prompt.